<h1>Binary Analysis & Symbolic Execution</h1>

<h2>Section 1: Hash Calculation</h2>

In [5]:
import hashlib
# Function that creates a hash for a given file
def compute_hash(path, algorithm):
    # Creates a new hash object using an algorithm
    h = hashlib.new(algorithm)
    # opens the file in bianry mdode and reads the
    # entire file and updates the hash object with its content and returns a 
    # final hash value
    with open(path, "rb") as f:
        h.update(f.read())
    return h.hexdigest()
# path to the executable file being analysed
# Procmon.exe is a safe Sysinternals tool used for static analysis
sample = r"C:\Users\emelz\Downloads\ProcessMonitor\Procmon.exe"
print("MD5: ", compute_hash(sample, "md5"))
print("SHA1: ", compute_hash(sample, "sha1"))
print("SHA256:", compute_hash(sample, "sha256"))

MD5:  c3e77b6959cc68baee9825c84dc41d9c
SHA1:  bc18a67ad4057dd36f896a4d411b8fc5b06e5b2f
SHA256: 3b7ea4318c3c1508701102cf966f650e04f28d29938f85d74ec0ec2528657b6e


<h2>Section 2: String Extraction</h2>

In [6]:
import re 
 
#  Function to extract an ASCII string from a binary file 
# This uses the string extraction method in static malware analysis
def extract_strings(path): 
    with open(path, "rb") as f: 
        data = f.read() 

# Regex patterns that matches any sequence of ascii patterns, this is a 
# common method used to find meaningful strings
    pattern = rb"[ -~]{4,}" 
    return re.findall(pattern, data) 
 # extract strings from the sample executable
strings = extract_strings(sample) 
 # only a small sample is shown to avoid overwhelming output
for s in strings[:20]: 
    print(s.decode(errors="ignore")) 

!This program cannot be run in DOS mode.
V*0T
0RichU
.text
`.rdata
@.data
.rsrc
@.reloc
hpqQ
h`EN
h|nN
h\nN
hlnN
=UUU
h_rM
hDLN
h`GO
hDLN
h|GO
hDLN


<h2>Section 3: PE Header Inspection Using pefile</h2>

In [9]:
import pefile 
 
# Loads portable executable file to allow for inspection of the metadata 
# inside
pe = pefile.PE(sample) 
#  prints the entry point of the program which is where
# the execution will begin and the program is loaded
print("Entry Point:", hex(pe.OPTIONAL_HEADER.AddressOfEntryPoint)) 
# prints the image base address which is the 
# preferred memory address where the program is to be loaded.
print("Image Base:", hex(pe.OPTIONAL_HEADER.ImageBase)) 
 
print("\nImported DLLs and functions:") 
# lists all external libaries the file uses 
for entry in pe.DIRECTORY_ENTRY_IMPORT: 
    print(" ", entry.dll.decode()) 
    for imp in entry.imports[:5]: 
        print("     -", imp.name.decode() if imp.name else "None") 

Entry Point: 0xa7f70
Image Base: 0x400000

Imported DLLs and functions:
  WS2_32.dll
     - getsockname
     - listen
     - recv
     - closesocket
     - socket
  VERSION.dll
     - GetFileVersionInfoW
     - VerQueryValueW
     - GetFileVersionInfoSizeW
  COMCTL32.dll
     - ImageList_ReplaceIcon
     - ImageList_SetBkColor
     - ImageList_AddMasked
     - ImageList_BeginDrag
     - ImageList_EndDrag
  FLTLIB.DLL
     - FilterSendMessage
     - FilterGetMessage
     - FilterReplyMessage
     - FilterConnectCommunicationPort
  KERNEL32.dll
     - AcquireSRWLockExclusive
     - AcquireSRWLockShared
     - InitializeSRWLock
     - GetSystemInfo
     - VerSetConditionMask
  USER32.dll
     - LoadStringA
     - DrawEdge
     - GetMessageW
     - TranslateMessage
     - DispatchMessageW
  GDI32.dll
     - SaveDC
     - RestoreDC
     - SetBrushOrgEx
     - SetPixel
     - PatBlt
  COMDLG32.dll
     - ChooseColorW
     - GetOpenFileNameW
     - PrintDlgW
     - ChooseFontW
     - FindText

<h2>Section 4: YARA Analysis</h2>

In [ ]:
import yara 
 
#  Triggers whenever the string http appears anywhere inside the file
rule_source = """ 
rule ContainsHTTP { 
    strings: 
        $s = "http" 
    condition: 
        $s 
} 
""" 
 
rules = yara.compile(source=rule_source) 
matches = rules.match(sample) 
print(matches)

[ContainsHTTP]


<h2>Section 5: Complete Static Triage Workflow</h2>

In [ ]:
import hashlib, pefile, re, yara 
 
# Hashes files to detect changes 
def compute_hashes(path): 
    algos = ["md5", "sha1", "sha256"] 
    output = {} 
    for a in algos: 
        h = hashlib.new(a) 
        with open(path, "rb") as f: 
            h.update(f.read()) 
        output[a] = h.hexdigest() 
    return output 
#  Extracts the ASCII strings
def extract_strings(path): 
    with open(path, "rb") as f: 
        data = f.read() 
    return re.findall(rb"[ -~]{4,}", data) 
 
print("Hashes:", compute_hashes(sample)) 
 
print("\nStrings:") 
print(extract_strings(sample)[:10]) 
 
print("\nImports:") 
# imports the tables to help find out what the program is doing 
# and what malicious functions have been imported.
pe = pefile.PE(sample) 
for entry in pe.DIRECTORY_ENTRY_IMPORT: 
    print(entry.dll.decode()) 
 
#  Scans the decoded binary fir URLS and addresses
print("\nIOCs:") 
decoded = open(sample, "rb").read().decode(errors="ignore") 
print("URLs:", re.findall(r"https?://[^\s\"']+", decoded)) 
print("IPs:", re.findall(r"\b\d{1,3}(?:\.\d{1,3}){3}\b", decoded)) 
 
print("\nYARA:") 
# Applies a simple yara rule which triggers if the file contains http
rule = yara.compile(source=""" 
rule Simple { 
    strings: $s = "http" 
    condition: $s 
} 
""") 
print(rule.match(sample))

Hashes: {'md5': 'c3e77b6959cc68baee9825c84dc41d9c', 'sha1': 'bc18a67ad4057dd36f896a4d411b8fc5b06e5b2f', 'sha256': '3b7ea4318c3c1508701102cf966f650e04f28d29938f85d74ec0ec2528657b6e'}

Strings:
[b'!This program cannot be run in DOS mode.', b'V*0T', b'0RichU', b'.text', b'`.rdata', b'@.data', b'.rsrc', b'@.reloc', b'hpqQ', b'h`EN']

Imports:
WS2_32.dll
VERSION.dll
COMCTL32.dll
FLTLIB.DLL
KERNEL32.dll
USER32.dll
GDI32.dll
COMDLG32.dll
ADVAPI32.dll
SHELL32.dll
ole32.dll
OLEAUT32.dll
SHLWAPI.dll
UxTheme.dll
dwmapi.dll
ntdll.dll

IOCs:
URLs: ['https://go.microsoft.com/fwlink/?LinkId=521839', 'https://go.microsoft.com/fwlink/?LinkId=521839', 'https://go.microsoft.com/fwlink/?LinkId=521839\\ul0\\cf0}}}}\\f0\\fs20', 'https://microsoft.com/exporting', 'https://microsoft.com/exporting}}}}\\f0\\fs19', 'http://www.microsoft.com/pkiops/crl/Microsoft%20Windows%20Third%20Party%20Component%20CA%202012.crl0\x06\x08+\x06\x01\x05\x05\x07\x01\x01\x04u0s0q\x06\x08+\x06\x01\x05\x05\x070\x02ehttp://www.microso

<h2>(Challenge) Section 6: Http strings</h2>

In [ ]:
import re

path = r"C:\Users\emelz\Downloads\ProcessMonitor\Procmon.exe"

# opens file in binary mode
with open(path, "rb") as f:
    data = f.read()
# extracts ascii characters
strings = re.findall(rb"[ -~]{4,}", data)

print("suspicious strings found:\n")
# loops through all extracted strings and decodes them into text
for s in strings:
    text = s.decode(errors="ignore")
    # filters out 
    if "http" in text.lower() or ".exe" in text.lower():
        print(text)



suspicious strings found:

\pard\fi-357\li357\sb120\sa120\tx360\b\fs20 4.\tab DATA COLLECTION.  \b0 The Sysinternals tools do not collect any data. Please refer to the {{\field{\*\fldinst{HYPERLINK "https://go.microsoft.com/fwlink/?LinkId=521839" }}{\fldrslt{Microsoft Privacy Statement\ul0\cf0}}}} <{{\field{\*\fldinst{HYPERLINK "https://go.microsoft.com/fwlink/?LinkId=521839"}}{\fldrslt{https://go.microsoft.com/fwlink/?LinkId=521839\ul0\cf0}}}}\f0\fs20 >.\b\par
\caps\fs20 6.\tab\fs19 Export Restrictions\caps0 .\b0   The software is subject to United States export laws and regulations.  You must comply with all domestic and international export laws and regulations that apply to the software.  These laws include restrictions on destinations, end users and end use.  For additional information, see {\cf1\ul{\field{\*\fldinst{HYPERLINK www.microsoft.com/exporting }}{\fldrslt{www.microsoft.com/exporting}}}}\cf1\ul\f0\fs19  <{{\field{\*\fldinst{HYPERLINK "https://microsoft.com/exporting"}}{\